In [ ]:
!pip install langchain_community langchain_openai langchain_huggingface pypdf faiss-cpu

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
#from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage,AIMessage
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

import os
from typing import List

In [7]:
import os
os.environ["OPENAI_API_KEY"] ="Your Key"

In [8]:
class Raglcel:
  def __init__(self, chunk_size = 400, chunk_overlap = 20):
    self.chunk_size = chunk_size,
    self.chunk_overlap = chunk_overlap,
    self.text_splitters = RecursiveCharacterTextSplitter(
        chunk_overlap = chunk_overlap,
        chunk_size = chunk_size
    )

  def processor(self, path_file :str)->List[Document]:
    lod = PyPDFLoader(path_file)
    data = lod.load()

    processod_chunks = []
    for i, docs in enumerate(data):
      cleaned_data = " ".join(docs.page_content.split())
      chunks = self.text_splitters.create_documents([cleaned_data])
      processod_chunks.extend(chunks)
    return processod_chunks


In [9]:
rag = Raglcel()
chunks = rag.processor("/content/Rama_V_MLE.pdf")

In [10]:
chunks

[Document(metadata={}, page_content='RAMA KRISHNA Denton, TX| ramadev4248@gmail.com | + 1 (940)-340-9284 | LinkedIn| GitHub EDUCATION University Of North Texas, Denton, TX January 2024 – May 2025 Master in Computer and Information Sciences GPA: 4.0/4.0 Relevant Coursework: Machine Learning, Artificial Intelligence, Distributed Systems, Database Management, Operating Systems. PROFESSIONAL SUMMARY AI/ML Engineer with a passion for'),
 Document(metadata={}, page_content='with a passion for crafting scalable, practical AI solutions, leveraging over four years of experience in big data and machine learning. I am an expert in agentic AI applications using LangChain, LangGraph, N8N, CrewAI, and GPT-4, with proficiency in Vertex AI, Big Query, and AWS SageMaker for model development and deployment. I am skilled in integrating data ecosystems with Chroma DB, Faiss, and'),
 Document(metadata={}, page_content='DB, Faiss, and Pinecone for efficient vector storage and retrieval. Experienced in buil

In [11]:
print(f"chunks are: {len(chunks)} ...")
print(f"content: {chunks[0].page_content[:100]}")
print(f"content: {chunks[1].page_content[:100]}")

chunks are: 18 ...
content: RAMA KRISHNA Denton, TX| ramadev4248@gmail.com | + 1 (940)-340-9284 | LinkedIn| GitHub EDUCATION Uni
content: with a passion for crafting scalable, practical AI solutions, leveraging over four years of experien


In [14]:
vector_store = FAISS.from_documents(
        documents = chunks,
        embedding = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2"),
        )
print(f"vectors : {vector_store.index.ntotal} vectors")

vectors : 18 vectors


In [15]:
vector_store.save_local("faiss-index")
print("vector stores in faiss directory")

vector stores in faiss directory


In [16]:
query = "what is fine-tuning"

results = vector_store.similarity_search(query, k=3)
print(results)

[Document(id='257ee55a-f9ca-4acb-8a0b-6b0aa30f9dbd', metadata={}, page_content='task tracking and timely delivery. TECHNICAL SKILLS \uf0b7 Programming Languages: Python, SQL, Java, PySpark. \uf0b7 Machine Learning: NumPy, Pandas, scikit-learn, XGBoost, Regression Analysis, Time Series, Clustering. \uf0b7 Gen AI: Gpt-4 series models, LangChain, LangGraph, MCP, Agentic AI workflows, fine-tuning (LoRA/QLoRA), crew AI, ReAct, N8N, Zapier, uvicorn, PydanticAI, Big Query. \uf0b7 Deep Learning & NLP:'), Document(id='b2df7600-80f0-4f7c-95d2-d3c1fb41dbd3', metadata={}, page_content='with a passion for crafting scalable, practical AI solutions, leveraging over four years of experience in big data and machine learning. I am an expert in agentic AI applications using LangChain, LangGraph, N8N, CrewAI, and GPT-4, with proficiency in Vertex AI, Big Query, and AWS SageMaker for model development and deployment. I am skilled in integrating data ecosystems with Chroma DB, Faiss, and'), Document(id='4c7

In [17]:
retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwarg = {"k":3}
    )

In [18]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x78d24cb33cb0>, search_kwargs={})

In [19]:
llm = ChatOpenAI(model = "gpt-4o-mini",temperature = 0)

In [92]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableParallel

In [93]:

my_prompt = ChatPromptTemplate.from_messages([
    ("system" ,  """
You are an assistant for question and answer task.
Use the following retrieved context to answer the question
If you don't know the answer say don't know
keep answer in 3 sentences and precise """),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\n\nQuestion: {question}"),
    ])

my_prompt


ChatPromptTemplate(input_variables=['context', 'question'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='System

In [94]:
def format_docs(docs):
   return "\n\n".join(doc.page_content for doc in docs)

In [95]:
def create_conversational_rag():
  return(
      RunnablePassthrough.assign(
          context = lambda x: format_docs(retriever.invoke(x["question"]))
      )
      |my_prompt
      |llm
      |StrOutputParser()
  )

conversational_rag = create_conversational_rag()

In [96]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['question'])))
})
| ChatPromptTemplate(input_variables=['context', 'question'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Ta

In [97]:
response = conversational_rag.invoke({"question" : "What you know about CI/CD"})
response


'CI/CD stands for Continuous Integration and Continuous Deployment, which are practices aimed at improving software development processes. Continuous Integration involves automatically testing and integrating code changes into a shared repository, while Continuous Deployment automates the release of these changes to production environments. Together, they enhance collaboration, reduce integration issues, and accelerate the delivery of high-quality software.'

In [98]:
chat_history = []

q1 = "what is CI/CD"
a1 = conversational_rag.invoke({"question" : q1,
                                "chat_histroy": chat_history})

print(q1)
print(a1)

what is CI/CD
CI/CD stands for Continuous Integration and Continuous Deployment. It is a set of practices that enable developers to integrate code changes frequently and automate the deployment process, ensuring that software can be reliably released at any time. This approach helps in improving software quality and accelerating the release of new features.


In [100]:
chat_history.extend([HumanMessage(content = q1),
                    AIMessage(content = a1)])

In [101]:
chat_history = []

q2 = "what is it diffrent"
a2 = conversational_rag.invoke({"question" : q2,
                                "chat_histroy": chat_history})

print(q1)
print(a2)

what is CI/CD
The context describes a project focused on developing an NLP-based answer checker, which involves designing a web platform and training a Python NLP model for improved answer similarity. It highlights the use of various technologies and tools, such as Figma, React, TensorFlow, and Pinecone, to enhance the system's performance and accuracy. The project emphasizes the integration of advanced data processing and machine learning techniques to optimize user experience and query resolution.
